# Interpolation Checks

Compare raw data against the processed interpolated outputs.


In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()
if DATA_DIR.name != 'Data':
    DATA_DIR = Path('Data')

RAW_DIR = DATA_DIR / 'Raw'
PROCESSED_DIR = DATA_DIR / 'Processed'

RAW_TFIM_1D = RAW_DIR / 'Data TFIM' / 'Data TFIM 1D'
RAW_TFIM_2D = RAW_DIR / 'Data TFIM' / 'Data TFIM 2D'
RAW_HEISENBERG = RAW_DIR / 'Data Heisenberg'
RAW_THERMAL_IPR = RAW_DIR / 'Data TFIM Thermal' / 'Data IPR Kernel'
RAW_THERMAL_HS = RAW_DIR / 'Data TFIM Thermal' / 'Data HS Kernel'

PROC_TFIM_1D = PROCESSED_DIR / 'DataSyn TFIM' / 'DataSyn TFIM 1D'
PROC_TFIM_2D = PROCESSED_DIR / 'DataSyn TFIM' / 'DataSyn TFIM 2D'
PROC_HEISENBERG = PROCESSED_DIR / 'DataSyn Heisenberg'
PROC_THERMAL_IPR = PROCESSED_DIR / 'DataSyn TFIM Thermal' / 'DataSyn IPR Kernel'
PROC_THERMAL_HS = PROCESSED_DIR / 'DataSyn TFIM Thermal' / 'DataSyn HS Kernel'
PROC_THERMAL_H_GRID = PROCESSED_DIR / 'DataSyn TFIM Thermal' / 'h_dense_4000.csv'

TFIM_1D_L_VALUES = list(range(3, 11)) + list(range(20, 101, 10))
THERMAL_L_VALUES = [20, 50, 100]
THERMAL_BETAS = [float(v) for v in np.round(np.arange(0.1, 2.0 + 1e-12, 0.1), 1)]

RAW_COLOR = 'crimson'
INTERP_COLOR = 'seagreen'
CURVE_STYLES = ['-', '--', ':', '-.']
RAW_MARKERS = ['o', 's', '^', 'D']

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'legend.frameon': False,
})

def read_vector(path, column=None):
    path = Path(path)
    try:
        return np.loadtxt(path, delimiter=',', dtype=float)
    except ValueError:
        df = pd.read_csv(path)
        if column is None:
            if df.shape[1] != 1:
                raise ValueError(f'{path} has multiple columns; pass column explicitly.')
            column = df.columns[0]
        return df[column].to_numpy(dtype=float)

def available_l_values(root, prefix='L='):
    values = []
    for folder in Path(root).glob(f'{prefix}*'):
        if not folder.is_dir():
            continue
        token = folder.name.split('=', 1)[1].split('x', 1)[0]
        if token.isdigit():
            values.append(int(token))
    return sorted(values)

def style_axis(ax, x_label, title):
    ax.set_yscale('log')
    ax.set_xlabel(x_label)
    ax.set_ylabel('IPR')
    ax.set_title(title)
    ax.legend(ncol=2)

def plot_ipr_overlay(title, x_label, x_raw, x_interp, raw_series, interp_series):
    fig, ax = plt.subplots(figsize=(6.8, 4.2), constrained_layout=True)
    for i, name in enumerate(raw_series):
        marker = RAW_MARKERS[i % len(RAW_MARKERS)]
        linestyle = CURVE_STYLES[i % len(CURVE_STYLES)]
        ax.scatter(
            x_raw,
            raw_series[name],
            s=22,
            marker=marker,
            color=RAW_COLOR,
            alpha=0.8,
            label=f'raw {name}',
        )
        ax.plot(
            x_interp,
            interp_series[name],
            lw=1.9,
            ls=linestyle,
            color=INTERP_COLOR,
            label=f'interp {name}',
        )
    style_axis(ax, x_label, title)
    plt.show()

def plot_raw_vs_interpolated_ipr(title, x_label, raw_root, proc_root, folder_name, x_file, curves, L_values):
    for L in L_values:
        folder = folder_name(L)
        raw_folder = raw_root / folder
        proc_folder = proc_root / folder
        if not raw_folder.exists() or not proc_folder.exists():
            print(f'Skipping {title} {folder}: missing raw or processed folder')
            continue

        x_raw = read_vector(raw_folder / x_file)
        x_interp = read_vector(proc_folder / x_file)
        raw_series = {}
        interp_series = {}
        for label, raw_file, proc_file in curves:
            raw_series[label] = read_vector(raw_folder / raw_file, column='value')
            interp_series[label] = read_vector(proc_folder / proc_file, column='value')

        plot_ipr_overlay(f'{title} {folder}', x_label, x_raw, x_interp, raw_series, interp_series)

def plot_heisenberg_raw_vs_interpolated():
    source_delta = np.linspace(0.0, 5.0, 51)
    curves = [('IPR X', 'IPR X.csv'), ('IPR XY', 'IPR XY.csv'), ('IPR Z', 'IPR Z.csv')]
    for L in available_l_values(PROC_HEISENBERG):
        raw_folder = RAW_HEISENBERG / f'L={int(L)}'
        proc_folder = PROC_HEISENBERG / f'L={int(L)}'
        if not raw_folder.exists() or not proc_folder.exists():
            print(f'Skipping Heisenberg L={L}: missing raw or processed folder')
            continue

        delta_interp = read_vector(proc_folder / 'Delta.csv')
        raw_series = {label: read_vector(raw_folder / file_name, column='value') for label, file_name in curves}
        interp_series = {label: read_vector(proc_folder / file_name, column='value') for label, file_name in curves}
        plot_ipr_overlay(f'Heisenberg L={L}', 'Delta', source_delta, delta_interp, raw_series, interp_series)

def beta_token(beta):
    return f'{float(beta):.1f}'


## TFIM 1D IPR

Plot raw points against interpolated curves for `L = 3, ..., 10` and then `L = 20, 30, ..., 100`.


In [ ]:
plot_raw_vs_interpolated_ipr(
    title='TFIM 1D',
    x_label='h',
    raw_root=RAW_TFIM_1D,
    proc_root=PROC_TFIM_1D,
    folder_name=lambda L: f'L={int(L)}x1',
    x_file='hs.csv',
    curves=[('IPR X', 'ipr_x.csv', 'ipr_x.csv'), ('IPR Z', 'ipr_z.csv', 'ipr_z.csv')],
    L_values=TFIM_1D_L_VALUES,
)


## TFIM 2D IPR

Plot raw points against interpolated curves for every available 2D TFIM system size.


In [ ]:
tfim_2d_l_values = available_l_values(PROC_TFIM_2D)
plot_raw_vs_interpolated_ipr(
    title='TFIM 2D',
    x_label='h',
    raw_root=RAW_TFIM_2D,
    proc_root=PROC_TFIM_2D,
    folder_name=lambda L: f'L={int(L)}x{int(L)}',
    x_file='hs.csv',
    curves=[('IPR X', 'ipr_x.csv', 'ipr_x.csv'), ('IPR Z', 'ipr_z.csv', 'ipr_z.csv')],
    L_values=tfim_2d_l_values,
)


## Heisenberg IPR

Plot raw points against interpolated curves for every available Heisenberg system size.


In [ ]:
plot_heisenberg_raw_vs_interpolated()


## Thermal TFIM IPR

For the interpolated thermal sizes `L = 20, 50, 100`, plot raw versus interpolated `IPR X` and `IPR Z` for each processed beta value.


In [ ]:
def plot_thermal_ipr_raw_vs_interpolated(L_values=THERMAL_L_VALUES, beta_values=THERMAL_BETAS):
    for L in L_values:
        raw = pd.read_csv(RAW_THERMAL_IPR / f'ipr_thermal_L={int(L)}.csv')
        proc = pd.read_csv(PROC_THERMAL_IPR / f'ipr_thermal_L={int(L)}.csv')
        for beta in beta_values:
            raw_b = raw[np.isclose(raw['beta'], beta)].sort_values('h')
            proc_b = proc[np.isclose(proc['beta'], beta)].sort_values('h')
            if raw_b.empty or proc_b.empty:
                print(f'Skipping thermal IPR L={L}, beta={beta}: missing raw or processed rows')
                continue

            raw_series = {
                'IPR X': raw_b['ipr_x'].to_numpy(dtype=float),
                'IPR Z': raw_b['ipr_z'].to_numpy(dtype=float),
            }
            interp_series = {
                'IPR X': proc_b['ipr_x'].to_numpy(dtype=float),
                'IPR Z': proc_b['ipr_z'].to_numpy(dtype=float),
            }
            plot_ipr_overlay(
                f'Thermal TFIM L={L}, beta={beta_token(beta)}',
                'h',
                raw_b['h'].to_numpy(dtype=float),
                proc_b['h'].to_numpy(dtype=float),
                raw_series,
                interp_series,
            )

plot_thermal_ipr_raw_vs_interpolated()


## Thermal TFIM HS Gram Matrices

For `L = 20, 50, 100`, compare the raw HS Gram matrix with the processed interpolated Gram matrix for each processed beta. The processed `4000 x 4000` matrix is downsampled only for plotting.


In [ ]:
def raw_hs_matrix(raw_df, beta):
    sub = raw_df[np.isclose(raw_df['beta'], beta)].copy()
    if sub.empty:
        return None, None, None
    h_a = np.sort(sub['h_a'].unique().astype(float))
    h_b = np.sort(sub['h_b'].unique().astype(float))
    K = sub.pivot(index='h_a', columns='h_b', values='kernel_real').reindex(index=h_a, columns=h_b).to_numpy(dtype=float)
    return h_a, h_b, K

def plot_thermal_hs_raw_vs_interpolated(L_values=THERMAL_L_VALUES, beta_values=THERMAL_BETAS, stride=20):
    h_dense = np.loadtxt(PROC_THERMAL_H_GRID, delimiter=',')
    h_plot = h_dense[::stride]

    for L in L_values:
        raw_df = pd.read_csv(RAW_THERMAL_HS / f'HS Kernel L={int(L)}.csv')
        for beta in beta_values:
            h_a, h_b, K_raw = raw_hs_matrix(raw_df, beta)
            proc_path = PROC_THERMAL_HS / f'L={int(L)}' / f'HS_kernel_beta={beta_token(beta)}.npy'
            if K_raw is None or not proc_path.exists():
                print(f'Skipping thermal HS L={L}, beta={beta}: missing raw rows or processed file')
                continue

            K_proc = np.load(proc_path, mmap_mode='r')[::stride, ::stride]

            fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
            im0 = axes[0].imshow(
                K_raw,
                origin='lower',
                extent=[h_b.min(), h_b.max(), h_a.min(), h_a.max()],
                aspect='auto',
            )
            axes[0].set_title(f'Raw HS L={L}, beta={beta_token(beta)}')
            axes[0].set_xlabel('h_b')
            axes[0].set_ylabel('h_a')
            fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

            im1 = axes[1].imshow(
                K_proc,
                origin='lower',
                extent=[h_plot.min(), h_plot.max(), h_plot.min(), h_plot.max()],
                aspect='auto',
            )
            axes[1].set_title(f'Interpolated HS L={L}, beta={beta_token(beta)}')
            axes[1].set_xlabel('h_b')
            axes[1].set_ylabel('h_a')
            fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
            plt.show()

plot_thermal_hs_raw_vs_interpolated()
